# Breast Cancer Fine-Tune — Mistral 7B with QLoRA

**Author:** DiegoDomLarr  
**Goal:** Specialize Mistral-7B-Instruct on breast cancer medical Q&A using QLoRA.

We go step by step:
1. Understand and prepare the data
2. Load the model in 4-bit (QLoRA)
3. Apply LoRA adapter
4. Train
5. Evaluate vs base model
6. Publish to HuggingFace

---

## Session Setup

Run this cell every time you open a new Colab session. It mounts Google Drive (for checkpoint safety) and authenticates with HuggingFace.

> **Before running:** Add your HF write token to Colab Secrets → left sidebar → 🔑 icon → name it `HF_TOKEN`.

In [11]:
import os, getpass
from google.colab import drive
from huggingface_hub import login

# Mount Google Drive — checkpoints land here so training survives a disconnect
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/FineTuneBreastCancer'
os.makedirs(f'{DRIVE_ROOT}/adapter', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/logs', exist_ok=True)

# Load HF token — reads from env first (Colab Secrets sets it automatically),
# falls back to manual paste if not found
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HF write token: ')

login(token=HF_TOKEN, add_to_git_credential=False)
print('Drive mounted and HuggingFace login done')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted and HuggingFace login done


## Step 1 — Understand the Data

Before touching a model, we need to understand what we're training on.

We're using **PubMedQA** — a dataset of real biomedical questions derived from PubMed abstracts, with human-verified answers. It lives on HuggingFace and we can load it in one line.

Each example has:
- `question` — the biomedical question
- `context` — the PubMed abstract (paragraphs + MeSH tags)
- `long_answer` — the prose answer we want the model to learn
- `final_decision` — yes / no / maybe

We'll also pull from a second dataset (`lavita/ChatDoctor-HealthCareMagic-100k`) to get more breast cancer examples — because PubMedQA alone only gives us ~30 after filtering.

In [ ]:
# Install the libraries we need for data loading
!pip install datasets -q

In [ ]:
from datasets import load_dataset

# Load PubMedQA — the labeled split has 1000 human-verified examples
pubmedqa = load_dataset("qiaojin/PubMedQA", "pqa_labeled")

print(pubmedqa)
print("\n--- One example ---")
ex = pubmedqa["train"][0]
print("Question:", ex["question"])
print("Long answer:", ex["long_answer"][:300])
print("Decision:", ex["final_decision"])
print("MeSH tags:", ex["context"]["meshes"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})

--- One example ---
Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Long answer: Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of our knowledge, this is the first report of mitochondria and chloroplasts moving on transvacuolar str
Decision: yes
MeSH tags: ['Alismataceae', 'Apoptosis', 'Cell Differentiation', 'Mitochondria', 'Plant Leaves']


### Filter for breast cancer

We search across the question, answer, and MeSH tags for breast cancer keywords.

**Problem we discovered:** PubMedQA labeled only gives ~30 breast cancer examples after filtering — too few to fine-tune.

**Solution:** We also load `ChatDoctor-HealthCareMagic-100k`, a large patient Q&A dataset, and filter that for breast cancer too. Combined, we should have 200–400 examples — enough for a meaningful fine-tune.

In [ ]:
# Keywords that indicate breast cancer content
BC_KEYWORDS = [
    "breast cancer", "breast carcinoma", "breast tumour", "breast tumor",
    "BRCA1", "BRCA2", "HER2", "tamoxifen", "mastectomy", "lumpectomy",
    "mammogram", "ductal carcinoma", "lobular carcinoma", "triple negative breast",
    "aromatase inhibitor", "trastuzumab"
]

def is_breast_cancer(text):
    text = text.lower()
    return any(kw.lower() in text for kw in BC_KEYWORDS)

# Filter PubMedQA
def filter_pubmedqa(example):
    combined = example["question"] + " " + example["long_answer"] + " " + " ".join(example["context"]["meshes"])
    return is_breast_cancer(combined)

bc_pubmedqa = pubmedqa["train"].filter(filter_pubmedqa)
print(f"PubMedQA breast cancer examples: {len(bc_pubmedqa)}")

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

PubMedQA breast cancer examples: 29


In [ ]:
# Load the second dataset — patient Q&A from real doctor consultations
chatdoctor = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")

print(chatdoctor)
print("\n--- One example ---")
print(chatdoctor[0])

README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…):   0%|          | 0.00/70.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 112165
})

--- One example ---
{'instruction': "If you are a doctor, please answer the medical questions based on the patient's description.", 'input': 'I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!', 'output': 'Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In this condition, t

In [ ]:
# Filter ChatDoctor for breast cancer
def filter_chatdoctor(example):
    combined = (example.get("input", "") + " " + example.get("output", "") + " " + example.get("instruction", ""))
    return is_breast_cancer(combined)

bc_chatdoctor = chatdoctor.filter(filter_chatdoctor)
print(f"ChatDoctor breast cancer examples: {len(bc_chatdoctor)}")

Filter:   0%|          | 0/112165 [00:00<?, ? examples/s]

ChatDoctor breast cancer examples: 1032


### Unify into a single format

Both datasets have different field names. We normalize them to a simple structure:
```
{"question": "...", "answer": "..."}
```
Then in the next step we'll convert this into the instruction format Mistral expects.

In [ ]:
import pandas as pd

rows = []

# From PubMedQA: question + long_answer
for ex in bc_pubmedqa:
    if len(ex["long_answer"].strip()) > 50:  # skip very short answers
        rows.append({"question": ex["question"], "answer": ex["long_answer"]})

# From ChatDoctor: instruction/input -> output
for ex in bc_chatdoctor:
    question = (ex.get("instruction", "") + " " + ex.get("input", "")).strip()
    answer = ex.get("output", "").strip()
    if len(question) > 20 and len(answer) > 50:
        rows.append({"question": question, "answer": answer})

df = pd.DataFrame(rows)
print(f"Total breast cancer Q&A pairs: {len(df)}")
print(f"Avg question length: {df['question'].str.len().mean():.0f} chars")
print(f"Avg answer length: {df['answer'].str.len().mean():.0f} chars")
df.head(3)

Total breast cancer Q&A pairs: 1061
Avg question length: 541 chars
Avg answer length: 621 chars


,question,answer
0,Does HER2 immunoreactivity provide prognostic ...,HER2 immunoreactivity might have a limited pro...
1,Does immediate breast reconstruction compromis...,We found no evidence that IBR compromised the ...
2,Is the combination with 2-methoxyestradiol abl...,2ME is able to enhance the antiproliferative a...


### Push dataset to HuggingFace

Instead of saving a local JSON that disappears when the Colab session ends, we push the dataset directly to HuggingFace Datasets Hub.

This means any future session (or anyone else) can load it in one line:
```python
load_dataset('DiegoDomLarr/breast-cancer-qa')
```

In [ ]:
from datasets import Dataset

hf_dataset = Dataset.from_pandas(df)

hf_dataset.push_to_hub(
    'DiegoDomLarr/breast-cancer-qa',
    token=HF_TOKEN,
    private=False
)

print(f'Pushed {len(hf_dataset)} examples to DiegoDomLarr/breast-cancer-qa')

### ✅ Step 1 Complete

Your breast cancer Q&A dataset is now live on HuggingFace:
👉 https://huggingface.co/datasets/DiegoDomLarr/breast-cancer-qa

**From any future session, load it with one line:**
```python
from datasets import load_dataset
ds = load_dataset('DiegoDomLarr/breast-cancer-qa')
```

Next: **Step 2 — Understanding QLoRA** (why it works, what the numbers mean)

---